# Week 10 Lab: Logistic Regression and Classification Evaluation

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/labs/lab-10-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this lab, you'll build and evaluate logistic regression models for two classification problems: predicting credit card default and supporting breast cancer diagnosis. You'll go beyond accuracy to judge each model by the mistakes it makes, then choose a classification threshold based on what those mistakes cost.

## 🎯 Learning Objectives
By the end of this lab, you will be able to:
- Build a logistic regression model with scikit-learn and interpret the direction of its coefficients
- Compare a model's accuracy to a baseline and explain why accuracy alone can mislead
- Evaluate classification models with a confusion matrix, precision, recall, F1-score, and ROC-AUC
- Choose a classification threshold based on the business cost of false positives and false negatives

## 📚 This Lab Reinforces
- **Chapter 23: Introduction to Logistic Regression**
- **Chapter 24: Evaluating Classification Models**
- **Week 10 Tuesday Slides: From Numbers to Categories**

## 🕐 Estimated Time & Structure
**Total Time:** 75 minutes
**Mode:** Work together in groups (2–4 students), submit individually

- **[0–25 min]** Guided review and practice with the Default dataset
- **[25–30 min]** Class discussion and Q&A
- **[30–70 min]** Independent challenges with the Breast Cancer Wisconsin dataset
- **[70–75 min]** Lab wrap-up and homework quiz preparation

You are encouraged to work through the lab with **2–4 classmates**: talk through the problems and help each other get past coding hurdles. However, **you must write your own code, and you will submit your own notebook and homework quiz.**

## 💡 Why This Matters
Classification models drive yes/no decisions everywhere: approving loans, flagging fraud, prioritizing medical follow-ups. A model that's "97% accurate" can still miss most of the cases that matter. This lab builds the habit every analyst needs: evaluate a model by the mistakes it makes and what those mistakes cost the business.

## Setup

We'll work with two datasets: the **Default** dataset from ISLP for guided practice (the same data from Tuesday's lecture), and the **Breast Cancer Wisconsin** dataset for the independent challenges.

The `ISLP` package isn't installed in Colab by default, so run this cell first.

> ⏳ The install takes about a minute. If Colab asks you to **restart the session**, click **Restart**, then continue with the next cell (you don't need to rerun this one).

In [ ]:
%pip install -q ISLP

In [ ]:
# Required imports
import pandas as pd
import numpy as np
from ISLP import load_data
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

# Use this random state everywhere so your results match the homework quiz
RANDOM_STATE = 42

# Load the datasets
Default = load_data('Default')
cancer = pd.read_csv("https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/breast_cancer.csv")

print("Default dataset shape:", Default.shape)
print("Cancer dataset shape: ", cancer.shape)

## Part 1 — Guided Review: Predicting Credit Card Default (15 minutes)

Let's walk through the complete logistic regression workflow from Tuesday's lecture, this time adding a third feature: whether the customer is a **student**.

**📋 Workflow:**
1. Check the baseline: how common is the outcome?
2. Prepare the features and target, then split the data
3. Fit the model and read the coefficients
4. Predict probabilities for new customers
5. Evaluate: accuracy vs. baseline, confusion matrix, precision, recall, F1, and ROC-AUC

Run each cell and read the explanation that follows it.

In [ ]:
# Step 1: Check the baseline
Default.head()

In [ ]:
Default['default'].value_counts(normalize=True)

Only about **3%** of customers default. A "lazy" model that predicts *No* for everyone would be about **97% accurate** without learning anything. Keep that number in mind.

**Step 2: Prepare the data.** Two preparation steps you'll reuse in the challenges:

- **Convert the target to 1/0.** scikit-learn needs a numeric target, so `"Yes"` becomes `1`.
- **Dummy-encode categorical features.** `student` is `"Yes"`/`"No"`, so we turn it into a 0/1 column with `pd.get_dummies()`. We encode only the categorical column, then combine it with the numeric columns.

We also use `stratify=y` in the split. With a rare outcome like default, it guarantees the training and test sets get the same ~3% default rate.

In [ ]:
# Step 2: Prepare features and target, then split
X = pd.concat([
    Default[['balance', 'income']],
    pd.get_dummies(Default[['student']], drop_first=True).astype(int)
], axis=1)
y = (Default['default'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print(f"Features: {X.columns.tolist()}")
print(f"Training set: {len(X_train):,} rows | default rate {y_train.mean():.1%}")
print(f"Test set:     {len(X_test):,} rows | default rate {y_test.mean():.1%}")

In [ ]:
# Step 3: Fit the model and look at the coefficients
model = LogisticRegression()
model.fit(X_train, y_train)

pd.DataFrame({'feature': X.columns, 'coefficient': model.coef_[0]})

**Reading the coefficients** (focus on the **sign**):

- **`balance` is positive:** higher balances mean higher default risk
- **`income` is tiny:** almost no effect once balance and student status are known
- **`student_Yes` is negative:** *holding balance and income constant*, students are **less** likely to default than non-students

Coefficients are changes in log-odds, which are hard to explain to a manager. To talk about risk, convert to probabilities:

In [ ]:
# Step 4: Predict probabilities for two customers who differ only in student status
customers = pd.DataFrame({
    'balance': [1500, 1500],
    'income': [40000, 40000],
    'student_Yes': [0, 1]
})

customers.assign(prob_default=model.predict_proba(customers)[:, 1].round(3))

Same balance and income: the non-student has about a **10%** chance of default and the student about **6%**. That's the negative `student_Yes` coefficient in plain English.

**Step 5: Evaluate the model.** Start with accuracy, and compare it to the lazy baseline:

In [ ]:
# Step 5a: Accuracy vs. the lazy baseline
y_pred = model.predict(X_test)   # uses a 50% threshold

print(f"Model accuracy:         {accuracy_score(y_test, y_pred):.1%}")
print(f"Lazy baseline accuracy: {1 - y_test.mean():.1%}")

In [ ]:
# Step 5b: The confusion matrix
cm = confusion_matrix(y_test, y_pred)

pd.DataFrame(
    cm,
    index=['Actual: No Default', 'Actual: Default'],
    columns=['Predicted: No Default', 'Predicted: Default']
)

In [ ]:
# Step 5c: Precision, recall, F1, and ROC-AUC
y_prob = model.predict_proba(X_test)[:, 1]   # ROC-AUC uses probabilities, not 0/1 predictions

print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.3f}")

**What these tell the bank:**

| Metric | Asks | Our model |
|--------|------|-----------|
| **Precision** | When we flag a customer, how often are we right? | About 2 out of 3 flags are real defaulters |
| **Recall** | Of the customers who default, how many do we catch? | Only about **1 in 3**: it misses most defaulters |
| **F1-score** | One number that balances precision and recall | Low, because recall is low |
| **ROC-AUC** | How well does the model **rank** risky customers above safe ones? (0.5 = random, 1.0 = perfect) | About 0.95: excellent ranking |

That last row is the key insight: the model **ranks** customers well, but the default 50% threshold is too strict to catch most defaulters. That's a threshold problem, not a model problem.

### 🧪 Practice Exercise — Move the Threshold

**Business Scenario:** The bank's risk team says missing a defaulter costs far more than double-checking a good customer. They want to see what happens if the model flags anyone with a **30% or higher** probability of default.

**Your Task:** Fill in the `___` blanks.

1. Turn the predicted probabilities into 0/1 predictions using a **0.3** threshold
2. Build the confusion matrix for the new predictions
3. Calculate precision and recall at the new threshold
4. Compare with the 50% threshold above: what went up, what went down, and which would the risk team prefer?

In [ ]:
# Step 1: Apply a 0.3 threshold to the probabilities from Step 5c
preds_30 = (y_prob >= ___).astype(int)

# Step 2: Confusion matrix (actual values first, then predictions)
confusion_matrix(___, ___)

In [ ]:
# Step 3: Precision and recall at the 0.3 threshold
print(f"Precision: {precision_score(___, ___):.3f}")
print(f"Recall:    {recall_score(___, ___):.3f}")

*Step 4 — Your comparison:*

## Class Discussion/Q&A (5 minutes)

**Discussion prompts:**
- The model is 97% accurate but misses most defaulters. How would you explain that to a bank manager?
- The model's ROC-AUC is excellent, but its recall at 50% is poor. How can both be true?
- Who should choose the threshold: the analyst or the business? What information do you need from them?

**Common blockers and clarifications:**
- `predict()` always uses a 50% threshold. To use a different one, start from `predict_proba(...)[:, 1]`
- Metric functions take the **actual values first**, then the predictions
- `roc_auc_score` needs **probabilities**; the other metrics need **0/1 predictions**

## Part 3 — Independent Challenges: Breast Cancer Diagnosis (40 minutes)

For the next three challenges, you'll build and evaluate models that support breast cancer diagnosis. The specific numerical results you get will be used to answer questions on this week's **Canvas homework quiz**.

* **The support decreases with each challenge.** Challenge 1 gives you starter code with blanks, Challenge 2 gives you an outline only, and Challenge 3 is on your own.
* **Work together, submit individually.** Talk through the challenges with your group and help each other get unstuck, but **write your own code**. You'll each submit your own notebook and quiz.
* **DO NOT USE AI** to generate code for you.
* Feel free to ask questions or seek help from the instructor. Periodically, we'll stop and walk through parts of a challenge together.
* **Important:** use the exact parameters given (`test_size=0.3`, `random_state=RANDOM_STATE`, `stratify=y`, `max_iter=5000`). These produce the exact numbers needed for the quiz. Steps marked **📝** feed directly into quiz questions.

### 🔬 About the Dataset

The **Breast Cancer Wisconsin (Diagnostic)** dataset describes 569 breast tissue samples taken by fine needle aspiration. For each sample, digitized images of the cell nuclei were measured on 10 characteristics (such as `radius`, `texture`, `area`, `concavity`, and `symmetry`), and each characteristic is summarized three ways:

- **`_mean`**: the average across all cells in the sample
- **`_se`**: the standard error of the measurements
- **`_worst`**: the average of the three largest (most severe) values

That gives **30 numeric features**. The target, `diagnosis`, is **`M`** (malignant, cancerous) or **`B`** (benign).

**Business Context:** A hospital wants a model to flag samples for additional review by a pathologist. The two kinds of errors have very different costs:

- **False Negative** (malignant tumor classified as benign): delayed treatment, estimated at **$50,000** per case, and potentially life-threatening
- **False Positive** (benign tumor flagged as malignant): an unnecessary follow-up biopsy, about **$2,000** per case

> ⚠️ **Two things to watch for:**
> - Some column names contain a space, such as `concave points_mean`. Copy column names exactly.
> - Pass `max_iter=5000` to `LogisticRegression()`. These features are on very different scales, and the model needs extra iterations to finish fitting. Without it you'll see a `ConvergenceWarning` and different results.

In [ ]:
# Preview the cancer dataset
cancer.head()

### Challenge 1 — Mean-Features Model

🧩 **Starter code provided:** fill in the `___` blanks and write the steps marked with comments.

**Business Question:** Using only the 10 `_mean` measurements, how well can a logistic regression model identify malignant tumors?

**Your Task:**

1. **Explore the data**: How many samples are there? What percentage of the samples are malignant vs. benign? Are there any missing values?
2. **Prepare the data**: Create a binary target (`1` = malignant, `0` = benign), and select the 10 features that end with `_mean` (provided).
3. **Split the data**: 70/30 train/test split with `random_state=RANDOM_STATE` and `stratify=y`. 📝 How many samples are in the training set?
4. **Fit the model**: Fit `LogisticRegression(max_iter=5000)` on the training data.
5. **Evaluate**: Build the confusion matrix and calculate accuracy, precision, recall, and F1-score on the test set. How many malignant tumors did the model miss?
6. **Interpret**: Is this model's accuracy impressive compared to the baseline you found in Step 1? For this hospital, which metric matters most, and why?

**Context:** You'll compare these results against Challenge 2, so keep your variable names (`model_mean`, `preds_mean`) as given.

In [ ]:
# Step 1: Explore the data
print(cancer.shape)

# What percentage of samples are malignant (M) vs. benign (B)?
cancer['diagnosis'].___(normalize=True)

In [ ]:
# Step 1 (continued): How many missing values are there in total?
cancer.isna().___().sum()

In [ ]:
# Step 2: Create the binary target and select the _mean features
y = (cancer['diagnosis'] == ___).astype(int)   # 1 = malignant

mean_features = [col for col in cancer.columns if col.endswith('_mean')]   # PROVIDED
X_mean = cancer[mean_features]

print(f"Number of _mean features: {len(mean_features)}")

In [ ]:
# Step 3: Split the data (70/30, random_state=RANDOM_STATE, stratify=y)
X_train_mean, X_test_mean, y_train, y_test = train_test_split(
    ___, ___, test_size=___, random_state=___, stratify=___
)

print(f"Training set: {len(X_train_mean)} samples")
print(f"Test set:     {len(X_test_mean)} samples")

In [ ]:
# Step 4: Fit the model on the TRAINING data
model_mean = LogisticRegression(max_iter=5000)
model_mean.fit(___, ___)

In [ ]:
# Step 5: Evaluate on the TEST data
preds_mean = model_mean.predict(___)

# Confusion matrix: rows are actual (benign, malignant), columns are predicted
confusion_matrix(y_test, preds_mean)

# Now calculate accuracy, precision, recall, and F1-score for preds_mean

*Step 6 — Your interpretation:*

### Challenge 2 — All-Features Model

📝 **Outline only:** the code cell lists each step as a comment. You write all the code.

**Business Question:** The pathology team asks whether using **all 30 measurements** gives a better model than the `_mean` features alone, and whether it would save the hospital money.

**Your Task:**

1. **Prepare the data**: Create a feature matrix with every column **except** `diagnosis`. 📝 How many features does it contain?
2. **Split the data**: Use the **same** settings as Challenge 1 (70/30, `random_state=RANDOM_STATE`, `stratify=y`). 📝 How many samples are in the training set? *(Because the settings match, these are the same patients as Challenge 1, so the comparison is fair.)*
3. **Fit the model**: `LogisticRegression(max_iter=5000)` on the training data.
4. **Evaluate**: Build the confusion matrix and calculate accuracy, precision, recall, F1-score, and ROC-AUC on the test set. 📝 How many false negatives (malignant tumors classified as benign) did the model make? 📝 What is the ROC-AUC (rounded to 3 decimal places)? *Remember: `roc_auc_score` needs the predicted **probabilities**.*
5. **Compare the models**: Put the accuracy, precision, recall, and F1-score for both models side by side. 📝 Which metric improved the most from the mean-features model to the all-features model?
6. **Cost analysis**: Calculate the total cost of errors for **both** models: `total cost = (false negatives × $50,000) + (false positives × $2,000)`. 📝 What is the total cost for the all-features model? 📝 Which model is more cost-effective?

**Context:** Record your numerical results carefully. Most of the homework quiz comes from this challenge.

In [ ]:
# Step 1: Create the feature matrix with all columns except 'diagnosis'

# Step 2: Split the data (same settings as Challenge 1)

# Step 3: Fit LogisticRegression(max_iter=5000) on the training data

# Step 4: Confusion matrix, accuracy, precision, recall, F1-score, and ROC-AUC

# Step 5: Compare both models' metrics side by side (a small DataFrame works well)

# Step 6: Total cost of errors for both models

### Challenge 3 — Choose the Threshold

🚀 **On your own:** no starter code. Use your all-features model from Challenge 2.

**Business Question:** Both models so far use the default 50% threshold. Missing a cancer costs 25 times more than an unnecessary biopsy, so the hospital asks: *what threshold should we use?*

**Your Task:**

1. **Sweep the threshold**: Using the all-features model's predicted probabilities on the test set, evaluate these thresholds: **0.5, 0.4, 0.3, 0.2, 0.1, and 0.05**. For each one, calculate the number of false negatives, the number of false positives, recall, precision, and total cost. Use the **same costs as Challenge 2**: `total cost = (false negatives × $50,000) + (false positives × $2,000)`.
2. **Find the best threshold**: 📝 Which threshold has the **lowest total cost**?
3. **Explain the pattern**: Total cost falls as the threshold drops, but it doesn't keep falling all the way to 0.05. Why not?
4. **Make a recommendation**: In a short paragraph written for the hospital's leadership, recommend a threshold. Include what it means for patients (how many cancers missed, how many extra biopsies) and one reason to be cautious before adopting it. *Hint: how many patients are in the test set?*

**Context:** Your answer to Step 2 appears on the homework quiz.

💡 **Hint:** Revisit the Practice Exercise in Part 1, and the "Threshold Is a Dial" code from Tuesday's lecture.

In [ ]:
# Your code here

*Step 3 — Why doesn't the lowest threshold win?*

*Step 4 — Your recommendation to hospital leadership:*

## 🎓 Lab Wrap-Up & Reflection

### ✅ What You Accomplished
In this lab, you practiced:
- Building logistic regression models and interpreting the direction of their coefficients
- Comparing accuracy against a baseline and recognizing when accuracy misleads
- Evaluating models with confusion matrices, precision, recall, F1-score, and ROC-AUC
- Translating model errors into business costs and choosing a threshold to minimize them

### 🤔 Reflection Questions
Take 2-3 minutes to consider:
- When would you trust accuracy as a metric, and when wouldn't you?
- Adding features improved the cancer model. Would more features always help? What could go wrong?
- If the cost of an unnecessary biopsy doubled, how would you expect the best threshold to change?

### 🔗 Connection to Course Goals
This lab moves from building a model to *deciding how to use it*. Choosing a metric and a threshold is a business decision, not just a technical one, and it's the step that determines whether a model actually helps the people who rely on it.

### 📋 Next Steps
- **Homework:** Complete the Canvas quiz using your numerical results from today's challenges, and upload your completed notebook (due Sunday)
- **Next Tuesday:** Decision trees, a model that captures non-linear patterns and interactions without a straight-line (or S-curve) assumption
- **Additional Practice:** Try the end-of-chapter exercises in Chapters 23 and 24

---
**💾 Save your work.** You'll upload this notebook with your quiz, so make sure every cell has been run and shows its output.

## 🚨 Troubleshooting & Common Issues

**Issue 1:** `ConvergenceWarning: lbfgs failed to converge`
- **Solution:** Use `LogisticRegression(max_iter=5000)`. The model needs more iterations to finish fitting. Results from a model that didn't converge won't match the quiz.

**Issue 2:** `KeyError: 'concave_points_mean'`
- **Solution:** The column name has a space, not an underscore: `'concave points_mean'`. Use `cancer.columns` to see the exact names.

**Issue 3:** `ValueError: could not convert string to float: 'M'`
- **Solution:** Your feature matrix still contains the `diagnosis` column. Drop it with `cancer.drop(columns='diagnosis')`.

**Issue 4:** My ROC-AUC looks too low, or matches my accuracy
- **Solution:** `roc_auc_score` needs probabilities: `model.predict_proba(X_test)[:, 1]`, not `model.predict(X_test)`.

**Issue 5:** My numbers don't match my groupmates'
- **Solution:** Check that you used `test_size=0.3`, `random_state=RANDOM_STATE`, `stratify=y`, and `max_iter=5000`, and that you passed the **actual values first** to each metric function.

**General Debugging Tips:**
- Check shapes with `.shape` before fitting models
- Fit on the **training** data; evaluate on the **test** data
- If a variable seems wrong, re-run the notebook from the top (Runtime → Run all) to clear out stale values